(sec:nlo)=
# Multi-photon interactions

Multi-photon interactions describe the nonlinear responses of a molecular system to the simultaneous absorption or emission of two or more photons. These processes arise from higher‑order terms in the light–matter interaction and are governed by electronic transition pathways that do not appear in linear spectroscopy. Within the Born–Oppenheimer and electric‑dipole approximations, multi-photon effects are characterized by frequency‑dependent response functions—such as second‑ and third‑order polarizabilities—that encode resonant enhancements and selection rules beyond those of one‑photon transitions.

VeloxChem provides efficient tools for computing these nonlinear response properties at the levels of time‑dependent density functional theory (TDDFT) and the complex polarization propagator (CPP) aproach. By solving perturbation‑dependent response equations for multiple field frequencies, the program enables the evaluation of two‑photon absorption amplitudes, hyperpolarizabilities, and other nonlinear optical indicators relevant to multi-photon spectroscopies.

(sec:tpa)=
## Two-photon absorption

### TPA strengths

In the quadratic response theory framework, two‑photon absorption (TPA) spectra are determined from residues of the second‑order response function at half the excitation energy. TPA strengths factorize into products of transition moments, which quantify the effective coupling of the ground state to an excited state via the simultaneous interaction with two photons. The two-photon transition moment is given by the sum-over-states expression

$$
S^{f\leftarrow 0}_{\alpha\beta} = \frac{1}{\hbar}
\sum_{n} 
\Bigg[
    \frac{ 
         \langle 0 | \hat{\mu}_\alpha| n \rangle
         \langle n |\hat{\mu}_\beta | f \rangle  
         }{
         \omega_{n0}-\omega_{f0}/2
         } 
    +
    \frac{
        \langle 0 | \hat{\mu}_\beta| n \rangle 
        \langle n |\hat{\mu}_\alpha | f \rangle
        }{
        \omega_{n0}-\omega_{f0}/2
        }
    \Bigg]
$$

where $\hbar \omega_{n0}$  is the excitation energy for state $|n\rangle$, $\hat{\mu}_\alpha$ is the electric dipole moment operator along the Cartesian axis $\alpha$ , and the summation includes the ground state, $| n \rangle = | 0 \rangle$.

For an isotropic sample, the experimentally relevant TPA strengths are obtained from the orientational average of the squared magnitudes of the Cartesian transition moments. This averaging yields a scalar measure that captures the laser field polarization, and it provides the quantity directly comparable to measured TPA cross sections.

$$
\delta_f^\mathrm{TPA} = 
\frac{1}{15}
\sum_{\alpha, \beta}
\left(
2 S^{f\leftarrow 0}_{\alpha\beta} \big[S^{f\leftarrow 0}_{\alpha\beta}\big]^* +
S^{f\leftarrow 0}_{\alpha\alpha} \big[S^{f\leftarrow 0}_{\beta\beta}\big]^*
\right)
$$

We have here assumed an experimental single-beam, monochromatic, setup with linear polarization.

TPA simulations are sensitive toward the choice of basis set and exchange-correlation functional. The inclusion of diffuse functions in the basis set is typically required, and in regard with functionals, it has been demonstrated that the range-separated hybrid CAM-B3LYP and the hybrid meta-GGA MN15 functionals are good choices, see {cite}`ahmadzadeh_2024`.

**Python script**

In [3]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("para-nitroaniline")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "cam-b3lyp"
scf_results = scf_drv.compute(molecule, basis)

tpa_drv = vlx.TpaTransitionDriver()

tpa_drv.nstates = 3

tpa_results = tpa_drv.compute(molecule, basis, scf_results)

Reading para-nitroaniline from PubChem...

Reference: S. Kim, J. Chen, T. Cheng, A. Gindulyte, J. He, S. He, Q. Li, B. A. Shoemaker, P. A. Thiessen, B. Yu, L. Zaslavsky, J. Zhang, E. E. Bolton, Nucleic Acids Res., 2025, 53, D1516-D1525.

Please double-check the compound since names may refer to more than one record.

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Kohn-Sham                                            
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Dir

In [39]:
photon_energies = tpa_results["photon_energies"]
delta_tpa_values = tpa_results["cross_sections"]["linear"]

print("State   Photon energy   TPA strength")
print(36 * "=")

idx = 0
for key, delta_tpa in delta_tpa_values.items():
    print(f"{idx+1:>3} {photon_energies[idx] * 27.2114:12.4f} eV {delta_tpa:12.4f} GM")
    idx += 1

State   Photon energy   TPA strength
  1       1.4728 eV       0.0001 GM
  2       1.9233 eV       0.0011 GM
  3       2.0913 eV      25.7858 GM


In [2]:
molecule.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

**Text file**

:::{code}
@jobs
task: response
@end

@method settings
xcfun: MN15
basis: def2-svpd
@end

@response
property: tpa transition
nstates: 10
@end

@molecule
charge: 0
multiplicity: 1
xyz:
...
@end
:::

In [ ]:
import numpy as np

S = tpa_results["transition_moments"][np.float64(-0.16052224281162478)].real

delta = 0

for a in range(3):
    for b in range(3):

        delta += 2 * S[a, b] * S[a, b] + S[a, a] * S[b, b]

delta /= 15

print(delta)

In [11]:
from IPython.display import display

display(tpa_results)

{'photon_energies': [np.float64(0.054123641172122976),
  np.float64(0.07067907445659634),
  np.float64(0.07685288036057693)],
 'transition_moments': {np.float64(-0.054123641172122976): array([[-0.00739333+0.j, -0.03094885+0.j,  0.1897825 +0.j],
         [-0.03094885+0.j, -0.0101611 +0.j,  0.03031015+0.j],
         [ 0.1897825 +0.j,  0.03031015+0.j,  0.01752854+0.j]]),
  np.float64(-0.07067907445659634): array([[-0.20448681+0.j,  0.58106767+0.j,  0.04547782+0.j],
         [ 0.58106767+0.j,  0.1999599 +0.j,  0.03600263+0.j],
         [ 0.04547782+0.j,  0.03600263+0.j,  0.00463514+0.j]]),
  np.float64(-0.07685288036057693): array([[ -2.60216057+0.j,  16.1975698 +0.j,   5.92725268+0.j],
         [ 16.1975698 +0.j, -87.23832988+0.j, -33.82523967+0.j],
         [  5.92725268+0.j, -33.82523967+0.j,  -5.61488017+0.j]])},
 'cross_sections': {'linear': {np.float64(-0.054123641172122976): np.float64(6.463392343357653e-05),
   np.float64(-0.07067907445659634): np.float64(0.0011040962786876328),
  

### TPA cross section